<a href="https://colab.research.google.com/github/Ronaktaakle/carrer-route-/blob/main/pythont.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text[:100000]
print("Text length:", len(text))

Text length: 100000


In [8]:
import re

text = text.lower()
text = re.sub(r'[^a-z\s]', '', text)


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

token_list = tokenizer.texts_to_sequences([text])[0]

seq_length = 5
sequences = []
for i in range(seq_length, len(token_list)):
    seq = token_list[i-seq_length:i+1]
    sequences.append(seq)

sequences = np.array(sequences)
X, y = sequences[:, :-1], sequences[:, -1]

print("Total words:", total_words)
print("Total sequences:", len(sequences))

Total words: 3080
Total sequences: 17888


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(total_words, 100, input_length=seq_length),
    LSTM(150, return_sequences=True),
    Dropout(0.2),
    LSTM(100),
    Dense(100, activation='relu'),
    Dense(total_words, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=128,
    callbacks=[early_stop]
)

Epoch 1/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - accuracy: 0.0368 - loss: 6.9202 - val_accuracy: 0.0383 - val_loss: 6.5761
Epoch 2/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.0410 - loss: 6.3759 - val_accuracy: 0.0386 - val_loss: 6.6557
Epoch 3/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 7s 67ms/step - accuracy: 0.0410 - loss: 6.3284 - val_accuracy: 0.0383 - val_loss: 6.7575
Epoch 4/20
112/112 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - accuracy: 0.0410 - loss: 6.2715 - val_accuracy: 0.0377 - val_loss: 6.7269


In [12]:
def generate_text(seed_text, next_words, model, tokenizer, seq_length):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=seq_length, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

print(generate_text("to be or not", 20, model, tokenizer, seq_length))
print(generate_text("once upon a time", 20, model, tokenizer, seq_length))

to be or not the the the the the the the the the the the the the the the the the the the the
once upon a time the the the the the the the the the the the the the the the the the the the the
